# 24 · Final 72 h Gray-Box Scientific Freeze

This notebook freezes the scientific interpretation of the completed 72 h Gray-Box programme before any 24 h or 3 h replication begins. It performs **no new modelling or tuning**. Its purpose is to make the evidence chain, allowed claims, limitations, and operational decision logic explicit and reproducible.

Primary question: **when should a multimodal solar-flare forecasting system issue a fusion forecast, degrade to a fallback, or withhold a forecast under temporal shift, uncertainty failure, and data outages?**


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib
import pandas as pd
import numpy as np

ROOT=Path('/home/abmoses2000')
OUT=ROOT/'graybox_aia72_final_scientific_freeze_v1_20261006'
if OUT.exists():
    raise SystemExit(f'Refusing to overwrite existing output: {OUT}')
OUT.mkdir(parents=True)

ART={
 'master':ROOT/'graybox_aia72_master_predictions_v2_20261005/graybox_aia72_master_predictions.csv.gz',
 'sharp':ROOT/'sharp72_predictions_frozen_20261002.csv.gz',
 'cross_cycle':ROOT/'graybox_aia72_cross_cycle_reliability_audit_20261006',
 'calibration_diag':ROOT/'graybox_aia72_calibration_diagnostics_v2_20261006',
 'applicability':ROOT/'graybox_aia72_applicability_shift_v1_20261006',
 'ar_event':ROOT/'graybox_aia72_ar_event_disjoint_v1_20261006',
 'real_outage':ROOT/'graybox_aia72_real_outage_replay_v1_20261006',
 'outage_structure':ROOT/'graybox_aia72_real_outage_structure_v1_20261006',
 'aia_fallback':ROOT/'graybox_aia72_aia_only_fallback_feasibility_v1_20261006',
 'rolling':ROOT/'graybox_aia72_rolling_operational_replay_v1_20261006',
}

missing=[k for k,p in ART.items() if not p.exists()]
if missing:
    raise ValueError(f'Missing required completed 72h artifact(s): {missing}')

def sha256(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for c in iter(lambda:f.read(4*1024*1024),b''): h.update(c)
    return h.hexdigest()

# Pull only completed, frozen result tables.
roll=pd.read_csv(ART['rolling']/'final_replay_summary.csv')
cad=json.loads((ART['rolling']/'cadence_audit.json').read_text())
uq=pd.read_csv(ART['rolling']/'past_only_uq_monitor.csv')
aia=pd.read_csv(ART['aia_fallback']/'branch_feasibility_summary.csv')
aia_uq=pd.read_csv(ART['aia_fallback']/'aia_uncertainty_feasibility.csv')
outage=pd.read_csv(ART['real_outage']/'real_outage_replay_summary.csv')
episode=pd.read_csv(ART['outage_structure']/'episode_structure_by_year.csv')
expo=pd.read_csv(ART['outage_structure']/'graybox_exposure_by_episode_severity.csv')

# ---------------------------
# Key frozen quantitative facts
# ---------------------------
facts=[]

def add(section,metric,value,scope,note=''):
    facts.append({'section':section,'metric':metric,'value':value,'scope':scope,'note':note})

for _,r in roll.iterrows():
    add('rolling_replay',f"{r['scenario']}::{r['role']}::availability",float(r.availability_rate),r.role)
    add('rolling_replay',f"{r['scenario']}::{r['role']}::normal_rate",float(r.normal_rate),r.role)
    add('rolling_replay',f"{r['scenario']}::{r['role']}::degraded_rate",float(r.degraded_rate),r.role)
    add('rolling_replay',f"{r['scenario']}::{r['role']}::tss",float(r.tss),r.role)

add('cadence','declared_issue_cadence_minutes',cad['declared_cadence_minutes'],'all')
add('cadence','median_within_harp_delta_minutes',cad['median_within_harp_delta_minutes'],'all')
add('cadence','fraction_exact_96_minutes',cad['fraction_exact_96_minutes'],'all')

for _,r in aia[aia.system.eq('aia_only')].iterrows():
    add('aia_only_feasibility',f"{r.regime}::tss",float(r.tss),r.regime)
    add('aia_only_feasibility',f"{r.regime}::recall",float(r.recall),r.regime)
    add('aia_only_feasibility',f"{r.regime}::average_precision",float(r.average_precision),r.regime)

for _,r in aia_uq.iterrows():
    add('aia_only_uq',f"{r.regime}::flare_conformal_coverage",float(r.aia_q90_flare_coverage),r.regime)
    add('aia_only_uq',f"{r.regime}::singleton_rate",float(r.aia_q90_singleton_rate),r.regime)

for _,r in episode.iterrows():
    add('real_outage_structure',f"{int(r.year)}::clustered_episode_fraction",float(r.clustered_episode_fraction),str(int(r.year)))
    add('real_outage_structure',f"{int(r.year)}::median_episode_span_hours",float(r.median_episode_span_hours),str(int(r.year)))
    add('real_outage_structure',f"{int(r.year)}::max_episode_span_hours",float(r.max_episode_span_hours),str(int(r.year)))

for _,r in expo.iterrows():
    add('real_outage_exposure',f"{r.episode_severity}::mapped_cases",int(r.mapped_graybox_cases),'mapped_graybox')

pd.DataFrame(facts).to_csv(OUT/'frozen_key_metrics.csv',index=False)

# ---------------------------
# Claim ledger
# ---------------------------
claims=[
 {
  'claim_id':'C01',
  'status':'SUPPORTED',
  'claim':'The 72 h Gray-Box system can preserve forecast availability during mapped AIA acquisition failures by routing from multimodal fusion to SHARP fallback when SHARP remains technically available.',
  'evidence':'Phase E1 + Phase G real-AIA-incident replay',
  'boundary':'Engineering robustness replay; not verified historical live-service outage evidence.'
 },
 {
  'claim_id':'C02',
  'status':'SUPPORTED',
  'claim':'Observed AIA acquisition/recovery failures are strongly temporally clustered rather than well represented by independent random missingness.',
  'evidence':'Phase E2 empirical outage episodes',
  'boundary':'Empirical episode IDs are deterministic reconstructions, not recovered production block IDs.'
 },
 {
  'claim_id':'C03',
  'status':'SUPPORTED',
  'claim':'The frozen AIA branch is not sufficiently stable across regimes to justify automatic AIA-only degraded issuance when SHARP is unavailable.',
  'evidence':'Phase F branch skill + flare-class conformal transfer',
  'boundary':'Negative feasibility result for this frozen 72 h branch; does not prove all future AIA-only systems are unusable.'
 },
 {
  'claim_id':'C04',
  'status':'SUPPORTED',
  'claim':'Predictive disagreement and entropy can remain useful within-regime selective-risk indicators while failing as standalone global/domain-shift detectors.',
  'evidence':'Phase C1 applicability/shift audit + rolling UQ monitor',
  'boundary':'Feature-space OOD was not rerun because original SHARP OOD artifacts were unavailable.'
 },
 {
  'claim_id':'C05',
  'status':'SUPPORTED',
  'claim':'Class-conditional conformal reliability degrades substantially under later temporal shift; earlier nominal coverage must not be interpreted as guaranteed under non-exchangeable future regimes.',
  'evidence':'Mondrian calibration + cross-cycle/2026 flare coverage + rolling UQ monitor',
  'boundary':'No claim of conformal coverage guarantee under temporal shift.'
 },
 {
  'claim_id':'C06',
  'status':'SUPPORTED',
  'claim':'Mapped region components in the evaluated chronological roles are disjoint from the training components, and the 2026 degradation persists at AR-level aggregation.',
  'evidence':'Phase D AR-disjoint audit',
  'boundary':'Exact event-disjoint and lead-time analysis remains provenance-limited.'
 },
 {
  'claim_id':'C07',
  'status':'SUPPORTED_WITH_CAUTION',
  'claim':'The completed replay is prospective-style and past-only in monitoring, not a true prospective deployment trial.',
  'evidence':'Phase G protocol; 72 h horizon plus 24 h reporting delay before outcomes become visible',
  'boundary':'Forecasts were generated retrospectively from frozen artifacts.'
 },
 {
  'claim_id':'C08',
  'status':'NOT_SUPPORTED',
  'claim':'The current 72 h system has demonstrated universal predictive superiority over the strongest constituent model.',
  'evidence':'2026 locked statistics and branch comparisons',
  'boundary':'The contribution is trust routing and operational reliability, not universal TSS dominance.'
 },
 {
  'claim_id':'C09',
  'status':'NOT_SUPPORTED',
  'claim':'The current project implements a first-principles MHD or PINN solar-physics model.',
  'evidence':'Current system uses physics-informed SHARP variables plus AIA image model outputs.',
  'boundary':'Do not overclaim physics modelling.'
 },
 {
  'claim_id':'C10',
  'status':'PROVENANCE_LIMITED',
  'claim':'Exact historical feature-space OOD and exact event-disjoint/lead-time claims are available.',
  'evidence':'Missing original feature-distance and event-link artifacts',
  'boundary':'Document limitation rather than reconstructing unsupported substitutes.'
 },
]
pd.DataFrame(claims).to_csv(OUT/'scientific_claim_ledger.csv',index=False)

# ---------------------------
# Operational architecture freeze
# ---------------------------
architecture={
 'horizon_hours':72,
 'nominal_issue_cadence_minutes':96,
 'states':{
   'NORMAL':{
      'condition':'SHARP+AIA technically available and frozen multimodal trust gates pass.',
      'action':'Issue fixed 0.5/0.5 SHARP+AIA fusion forecast.'
   },
   'DEGRADED':{
      'condition':'NORMAL is false but SHARP remains technically available.',
      'action':'Issue SHARP fallback and label forecast DEGRADED.'
   },
   'ABSTAIN':{
      'condition':'SHARP is technically unavailable.',
      'action':'Withhold forecast. Automatic AIA-only fallback is not enabled.'
   }
 },
 'uq_layers':['probability calibration','Mondrian class-conditional conformal sets',
              'seed disagreement','fusion entropy','SHARP-AIA probability gap',
              'rolling past-only reliability monitoring'],
 'outcome_visibility':'issue time + 72 h horizon + 24 h reporting delay',
 'monitor_feedback_into_policy':False
}
(OUT/'operational_architecture_freeze.json').write_text(json.dumps(architecture,indent=2)+'\n')

# Last rolling-UQ row frozen for easy manuscript use.
uq_last=uq.tail(1).copy()
uq_last.to_csv(OUT/'rolling_uq_endpoint.csv',index=False)

freeze={
 'status':'FINAL_72H_SCIENTIFIC_FREEZE_COMPLETE',
 'created_utc':datetime.now(timezone.utc).isoformat(),
 'horizon_hours':72,
 'new_model_fitting':False,
 'new_recalibration':False,
 'new_threshold_tuning':False,
 'new_policy_selection':False,
 'new_policy_redesign':False,
 'primary_scientific_contribution':'Experimentally validated issue/degrade/withhold trust routing under temporal shift, uncertainty degradation, and realistic AIA acquisition failures.',
 'primary_policy':'fallback_v3_q90',
 'core_interpretation':[
   'Operational reliability is not equivalent to raw discrimination.',
   'Uncertainty estimates can become overconfident under temporal/domain shift.',
   'Real missingness is strongly correlated in time and region, motivating fallback-aware rather than IID-missingness-only evaluation.',
   'SHARP is currently the justified degraded fallback; AIA-only automatic fallback is not supported.',
   'The 72 h framework is frozen before 24 h and 3 h replication.'
 ],
 'limitations':[
   'Not a true prospective deployment trial.',
   'Exact feature-space OOD rerun is provenance-limited.',
   'Exact event-disjoint/lead-time analysis is provenance-limited.',
   'No universal superiority claim over SHARP.',
   'No first-principles MHD/PINN claim.'
 ],
 'source_sha256':{
   'master_predictions':sha256(ART['master']),
   'sharp_predictions':sha256(ART['sharp']),
   'rolling_protocol':sha256(ART['rolling']/'protocol.json'),
   'aia_fallback_protocol':sha256(ART['aia_fallback']/'protocol.json'),
   'real_outage_protocol':sha256(ART['real_outage']/'protocol.json'),
   'outage_structure_protocol':sha256(ART['outage_structure']/'protocol.json')
 },
 'outputs':['frozen_key_metrics.csv','scientific_claim_ledger.csv','operational_architecture_freeze.json','rolling_uq_endpoint.csv']
}
(OUT/'freeze_protocol.json').write_text(json.dumps(freeze,indent=2)+'\n')

print('===== FINAL 72H SCIENTIFIC FREEZE COMPLETE =====')
print('\nOperational architecture:')
print(json.dumps(architecture,indent=2))
print('\nClaim ledger:')
print(pd.DataFrame(claims)[['claim_id','status','claim']].to_string(index=False))
print('\nLast rolling UQ point:')
print(uq_last.to_string(index=False))
print('\nOutputs:',OUT)


===== FINAL 72H SCIENTIFIC FREEZE COMPLETE =====

Operational architecture:
{
  "horizon_hours": 72,
  "nominal_issue_cadence_minutes": 96,
  "states": {
    "NORMAL": {
      "condition": "SHARP+AIA technically available and frozen multimodal trust gates pass.",
      "action": "Issue fixed 0.5/0.5 SHARP+AIA fusion forecast."
    },
    "DEGRADED": {
      "condition": "NORMAL is false but SHARP remains technically available.",
      "action": "Issue SHARP fallback and label forecast DEGRADED."
    },
    "ABSTAIN": {
      "condition": "SHARP is technically unavailable.",
      "action": "Withhold forecast. Automatic AIA-only fallback is not enabled."
    }
  },
  "uq_layers": [
    "probability calibration",
    "Mondrian class-conditional conformal sets",
    "seed disagreement",
    "fusion entropy",
    "SHARP-AIA probability gap",
    "rolling past-only reliability monitoring"
  ],
  "outcome_visibility": "issue time + 72 h horizon + 24 h reporting delay",
  "monitor_feedback_in

## Frozen transition to the next horizon

After this notebook is executed and archived, the 72 h architecture is considered frozen. Replication at 24 h should reuse the **scientific architecture** (calibration → UQ → applicability/shift → fallback/abstention → realistic outage replay → past-only rolling evaluation) but must **not automatically reuse 72 h thresholds, calibration parameters, conformal quantiles, or policy cutoffs**. Those are horizon-specific and must be developed from the appropriate earlier 24 h data only.
